In [89]:
## Required libraries
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import time

In [90]:
#base_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])
#raw_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\raw_data_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])

base_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])
#raw_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\raw_data_ex.csv', encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])


In [93]:
# Required functions

def preprocess(x):
    
    x["Arrival"]= pd.to_datetime(x["Arrival"]) 
    x["Departure"]= pd.to_datetime(x["Departure"]) 
    x["Last Name"] = x["Last Name"].astype(str)
    result = pd.merge(left = base_df, 
                  right = x, 
                  left_on = ['Arrival_Date', 'Departure_Date'], 
                  right_on =['Arrival', 'Departure'], 
                  how = 'right')
    matched_df = result[result.Reservation_No.notnull() == True]
    unmatched_df = result[result.Reservation_No.notnull() == False]
    
    base_test = matched_df["Client_Guestname_1"].str.strip()
    base_test = base_test.str.replace(',','')
    base_test = base_test.str.upper()
    
    raw_test = matched_df["Last Name"].str.strip()
    raw_test = raw_test.str.upper()
    # list conversion for fuzzy matching
    raw_list = raw_test.values.tolist()
    base_list = base_test.values.tolist()
    
    # Fuzzy matching
    possibilities = []
    for string in raw_list:
        #print(string)
        possibility = process.extractOne(string, base_list, scorer=fuzz.token_sort_ratio)
        possibilities.append(possibility)
    temp_df = pd.DataFrame(possibilities, columns = ['Match_list', 'Match_score'])
    temp_df['Actual_string'] = raw_list
    temp_df['Actual_string'] = temp_df['Actual_string'].astype(str)
    temp_df.columns = ['Match_list', 'Match_score','Actual_string']
    interm_2 = pd.merge(left = matched_df, right = temp_df, left_on = 'Last Name', right_on = 'Actual_string' , how = 'left')
    interm_3 = interm_2.append(unmatched_df) 
    threshold = 70
    interm_3['Match_status'] = interm_3['Match_score'].apply(lambda x: 'Y' if x > threshold else 'N')
    interm_3['Confirmation'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Reservation_No'], interm_3['Confirmation'])
    interm_3['Last Name'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Client_Guestname_1'], interm_3['Last Name'])
    raw_col = list(raw_df.columns)
    raw_col.append("Match_status")
    final_raw = interm_3[raw_col]
    final_raw = final_raw.drop_duplicates()
    final_raw.to_csv("C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\result_raw1.csv",mode="a",sep = ';',header=True,index=False)

reader = pd.read_csv("C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv", chunksize=1000, encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])
# reader = pd.read_csv("C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\raw_data_ex.csv",
#                      parse_dates=['Arrival','Departure','Pay Date'],
#                      chunksize=1000, 
#                      encoding = "ISO-8859-1") # chunksize depends with you colsize


start_time = time.time()
#[preprocess(r) for r in reader]
for r in reader:
    preprocess(r)
    print(r.shape)
    
end_time = time.time()
print(end_time - start_time)
# result = pd.merge(left = base_df, 
#                   right = raw_df, 
#                   left_on = ['Arrival_Date', 'Departure_Date'], 
#                   right_on =['Arrival', 'Departure'], 
#                   how = 'right')
# result
print('SUCCESS')

C:\Anaconda\lib\site-packages\pandas\core\frame.py:7138: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  sort=sort,


(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(168, 44)
141.61318922042847
SUCCESS


In [102]:
ed = pd.read_csv(r'C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv', encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date']) # chunksize depends with you colsize
ed.shape

(44168, 44)

In [81]:
## Execution

# base_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1")
# raw_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\raw_data_firstJan.csv', encoding = "ISO-8859-1")
base_df=pd.read_csv('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])


csv_url='C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv'
# use chunk size 500
c_size = 2000

for gm_chunk in pd.read_csv(csv_url, encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'], chunksize=c_size):
     print(gm_chunk.shape)

base_df.shape

(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(2000, 44)
(168, 44)


In [8]:
result = pd.merge(left = base_df, 
                  right = raw_df, 
                  left_on = ['Arrival_Date', 'Departure_Date'], 
                  right_on =['Arrival', 'Departure'], 
                  how = 'right',
                  indicator=True)

result.query('_merge != "both"')

,Display Case No_,Reservation_No,Arrival_Date,Departure_Date,Client_Guestname_1,Client_Guestname_2,Description,MuseID,Handbooking,ProcessNumber,...,Agency Addr2,Agency City,Agency State Code,Agency Zip,Agency Country Code,Property Phone,Payment ID,Cheque Number,Pay Date,_merge
3,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,E1171556,0,20200318,NaN,NaN,NaN,NaN,NaN,NaT,right_only
4,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,E1171556,0,20200318,NaN,NaN,NaN,NaN,NaN,NaT,right_only


In [9]:
matched_df = result[result.Reservation_No.notnull() == True]
unmatched_df = result[result.Reservation_No.notnull() == False]
matched_df

,Display Case No_,Reservation_No,Arrival_Date,Departure_Date,Client_Guestname_1,Client_Guestname_2,Description,MuseID,Handbooking,ProcessNumber,...,Agency Addr2,Agency City,Agency State Code,Agency Zip,Agency Country Code,Property Phone,Payment ID,Cheque Number,Pay Date,_merge
0,V008636023,240643223.0,2020-01-01,2020-01-06,"Geiselmann, Tina",NaN,NaN,ACCOR,0.0,135197342.0,...,E1171556,0,20200318,NaN,NaN,NaN,NaN,NaN,NaT,both
1,V008636023,240643223.0,2020-01-01,2020-01-06,"Geiselmann, Tina",NaN,NaN,ACCOR,0.0,135197342.0,...,E1171556,0,20200318,NaN,NaN,NaN,NaN,NaN,NaT,both
2,V008679777,243545049.0,2020-01-02,2020-01-06,"tao, fuyong",NaN,NaN,ACCOR,0.0,136899089.0,...,E1171556,0,20200318,NaN,NaN,NaN,NaN,NaN,NaT,both


In [25]:
result1.to_excel('C:\\Users\\svi02\\Documents\\misc\\test_match1.xlsx', 
                  sheet_name = 'Match',
                  header = True,
                  encoding='utf-8',
                  index=False)

In [10]:
matched_df
base_test = matched_df["Client_Guestname_1"].str.strip()
base_test = base_test.str.replace(',','')
base_test = base_test.str.upper()
#base_test = base_test.values[0]
base_test.head()

0    GEISELMANN TINA
1    GEISELMANN TINA
2         TAO FUYONG
Name: Client_Guestname_1, dtype: object

In [11]:
matched_df
raw_test = matched_df["Last Name"].str.strip()
raw_test = raw_test.str.upper()
#raw_test = str(raw_test.values[0])
raw_test.head()

0    GEISELMANN TINA MR
1     BROSDA MICHAEL MR
2         TAO FUYONG MR
Name: Last Name, dtype: object

In [12]:
raw_list = raw_test.values.tolist()
base_list = base_test.values.tolist()

In [13]:
#fuzz.partial_ratio(raw_test, base_test)

In [18]:
import time
possibilities = []
start_time = time.time()

for string in raw_list:
    #print(string)
    possibility = process.extractOne(string, base_list, scorer=fuzz.token_sort_ratio)
    possibilities.append(possibility)
end_time = time.time()
print(end_time - start_time)
temp_df = pd.DataFrame(possibilities)
temp_df

0.0


,0,1
0,GEISELMANN TINA,91
1,GEISELMANN TINA,31
2,TAO FUYONG,87


In [17]:
temp_df['Actual_string'] = raw_list
temp_df.columns = ['Match_list', 'Match_score', 'Actual_string']
temp_df

,Match_list,Match_score,Actual_string
0,GEISELMANN TINA,91,GEISELMANN TINA MR
1,GEISELMANN TINA,31,BROSDA MICHAEL MR
2,TAO FUYONG,87,TAO FUYONG MR


In [10]:
interm_2 = pd.merge(left = matched_df, right = temp_df, left_on = 'Last Name', right_on = 'Actual_string' , how = 'left')
interm_2

,Display Case No_,Reservation_No,Arrival_Date,Departure_Date,Client_Guestname_1,Client_Guestname_2,Description,MuseID,Handbooking,ProcessNumber,...,Agency State Code,Agency Zip,Agency Country Code,Property Phone,Payment ID,Cheque Number,Pay Date,Match_list,Match_score,Actual_string
0,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,MARX HELENA,57,BREANT HELENE MRS
1,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,MARX HELENA,57,BREANT HELENE MRS
2,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,MARX HELENA,57,BREANT HELENE MRS
3,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,MARX HELENA,57,BREANT HELENE MRS
4,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,MARX HELENA,57,BREANT HELENE MRS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4336845,V008680055,227710106,20191229,20200101,"YANG, JunQian",NaN,NaN,ACCOR,0,127172434,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,ANNE GROSSMANN,60,SEYMOUR ANNE MRS
4336846,V008680055,227710106,20191229,20200101,"YANG, JunQian",NaN,NaN,ACCOR,0,127172434,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,ANNE GROSSMANN,60,SEYMOUR ANNE MRS
4336847,V008680055,227710106,20191229,20200101,"YANG, JunQian",NaN,NaN,ACCOR,0,127172434,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,ANNE GROSSMANN,60,SEYMOUR ANNE MRS
4336848,V008680055,227710106,20191229,20200101,"YANG, JunQian",NaN,NaN,ACCOR,0,127172434,...,NaN,50668,DE,49 30 26962939,E1171556,0,20200318,ANNE GROSSMANN,60,SEYMOUR ANNE MRS


In [11]:
interm_3 = interm_2.append(unmatched_df) 
interm_3[['Reservation_No', 'Arrival_Date', 'Departure_Date', 'Match_score', 'Match_list']]

C:\Anaconda\lib\site-packages\pandas\core\frame.py:7138: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  sort=sort,


,Reservation_No,Arrival_Date,Departure_Date,Match_score,Match_list
0,237896670,20191230,20200101,57.0,MARX HELENA
1,237896670,20191230,20200101,57.0,MARX HELENA
2,237896670,20191230,20200101,57.0,MARX HELENA
3,237896670,20191230,20200101,57.0,MARX HELENA
4,237896670,20191230,20200101,57.0,MARX HELENA
...,...,...,...,...,...
4336845,227710106,20191229,20200101,60.0,ANNE GROSSMANN
4336846,227710106,20191229,20200101,60.0,ANNE GROSSMANN
4336847,227710106,20191229,20200101,60.0,ANNE GROSSMANN
4336848,227710106,20191229,20200101,60.0,ANNE GROSSMANN


In [12]:

threshold = 70
interm_3['Match_status'] = interm_3['Match_score'].apply(lambda x: 'Y' if x > threshold else 'N')
interm_3['Confirmation'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Reservation_No'], interm_3['Confirmation'])
interm_3['Last Name'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Client_Guestname_1'], interm_3['Last Name'])
interm_3[['Confirmation', 'Reservation_No', 'Last Name',  'Client_Guestname_1']]
  

,Confirmation,Reservation_No,Last Name,Client_Guestname_1
0,1912300621,237896670,BREANT HELENE MRS,"WANG, MINGWEI"
1,1912300621,237896670,BREANT HELENE MRS,"WANG, MINGWEI"
2,1912300621,237896670,BREANT HELENE MRS,"WANG, MINGWEI"
3,1912300621,237896670,BREANT HELENE MRS,"WANG, MINGWEI"
4,1912300621,237896670,BREANT HELENE MRS,"WANG, MINGWEI"
...,...,...,...,...
4336845,1912290663,227710106,SEYMOUR ANNE MRS,"YANG, JunQian"
4336846,1912290663,227710106,SEYMOUR ANNE MRS,"YANG, JunQian"
4336847,1912290663,227710106,SEYMOUR ANNE MRS,"YANG, JunQian"
4336848,1912290663,227710106,SEYMOUR ANNE MRS,"YANG, JunQian"


In [14]:
interm_3['Last name'] = np.where(interm_3['Match_status'] == 'Y', str(interm_3['Reservation_No']), str(interm_3['Last Name']))
interm_3[['Confirmation', 'Reservation_No', 'Last Name',  'Client_Guestname_1']]

MemoryError: 

In [20]:
interm_3.to_excel('C:\\Users\\svi02\\Documents\\misc\\test_match_2.xlsx', 
                  sheet_name = 'Match',
                  header = True,
                  encoding='utf-8',
                  index=False)

ValueError: This sheet is too large! Your sheet size is: 4336850, 101 Max sheet size is: 1048576, 16384

In [16]:
matched_df.shape

(8620, 97)

In [17]:
unmatched_df.shape

(0, 97)

In [18]:
base_df.shape

(1700, 53)

In [19]:
raw_df.shape

(20, 44)

In [26]:
base_df.head(15)

,Display Case No_,Reservation_No,Arrival_Date,Departure_Date,Client_Guestname_1,Client_Guestname_2,Description,MuseID,Handbooking,ProcessNumber,...,Agency Line Amount (LCY),TAF Line Amount,TAF Line Amount (LCY),TAF Type,TAF Rate,TAF Fix,TAF Contract Code,TAF Function ID,TAF Function Desc_,TAF Business Rules Code
0,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,7.67,0,0,0,0,0,NaN,0,NaN,NaN
1,K001116009,237896670,20191230,20200101,"WANG, MINGWEI",NaN,NaN,ACCOR,0,133525571,...,18.59,0,0,0,0,0,NaN,0,NaN,NaN
2,K001116009,242543996,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136320830,...,32.76,0,0,0,0,0,NaN,0,NaN,NaN
3,K001116009,242543996,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136320830,...,9.49,0,0,0,0,0,NaN,0,NaN,NaN
4,K001116009,242543996,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136320830,...,13.26,0,0,0,0,0,NaN,0,NaN,NaN
5,K001116009,242547440,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136322573,...,24.57,0,0,0,0,0,NaN,0,NaN,NaN
6,K001116009,242547440,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136322573,...,9.49,0,0,0,0,0,NaN,0,NaN,NaN
7,K001116009,242547440,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136322573,...,13.26,0,0,0,0,0,NaN,0,NaN,NaN
8,K001116009,242547440,20191226,20200101,"LINGODUN, ISLAM",NaN,NaN,ACCOR,0,136322573,...,12.61,0,0,0,0,0,NaN,0,NaN,NaN
9,K001116014,237000028,20191231,20200101,"van Lier, Peter",NaN,NaN,ACCOR,0,132987343,...,17.42,0,0,0,0,0,NaN,0,NaN,NaN
